In [66]:
from llama_index.core import SimpleDirectoryReader, Settings
from llama_index.embeddings.text_embeddings_inference import TextEmbeddingsInference
from llama_index.llms.openai_like import OpenAILike
import json
from pathlib import Path

In [67]:
COLLECT_NAME = "wb_oferta_pvz"
EMBED_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

In [68]:
# подключение embedding модели с CPU
Settings.embed_model = TextEmbeddingsInference(model_name=EMBED_MODEL_NAME,
                                               base_url="http://localhost:8081")

In [69]:
# специальные промпы для Qwen
def completion_to_prompt(completion):
   return f"<|im_start|>system\n<|im_end|>\n<|im_start|>user\n{completion}<|im_end|>\n<|im_start|>assistant\n"

def messages_to_prompt(messages):
    prompt = ""
    for message in messages:
        if message.role == "system":
            prompt += f"<|im_start|>system\n{message.content}<|im_end|>\n"
        elif message.role == "user":
            prompt += f"<|im_start|>user\n{message.content}<|im_end|>\n"
        elif message.role == "assistant":
            prompt += f"<|im_start|>assistant\n{message.content}<|im_end|>\n"

    if not prompt.startswith("<|im_start|>system"):
        prompt = "<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n" + prompt

    prompt = prompt + "<|im_start|>assistant\n"

    return prompt

In [70]:
# подключение qwen3-4b-fp8 llm из vLLM
# Settings.llm = OpenAILike(
#     messages_to_prompt=messages_to_prompt,
#     completion_to_prompt=completion_to_prompt,
#     model="qwen3-4b",                    
#     api_base="http://localhost:8000/v1", 
#     # api_key="dummy",                     
#     temperature=0.0,
#     max_tokens=512,                    
#     timeout=120,                        
#     additional_kwargs={
#         "top_p": 0.8,
#         "frequency_penalty": 0.0,
#         "presence_penalty": 0.0,
#     }
# )



In [71]:
# # подключение qwen3-14b-awq llm из vLLM
Settings.llm = OpenAILike(
    messages_to_prompt=messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    system_prompt="""Ты эксперт по договорным отношениям в России. 
                    Ты ВСЕГДА отвечаешь ТОЛЬКО на русском языке. 
                    Ты ВСЕГДА отвечаешь ТОЛЬКО в формате JSON без дополнительного текста.""",
    model="qwen3-14b",                    
    api_base="http://localhost:8000/v1", 
    # api_key="dummy",                     
    temperature=0.0,
    max_tokens=512,                    
    timeout=120,                        
    additional_kwargs={
        "top_p": 0.8,
        "frequency_penalty": 0.0,
        "presence_penalty": 0.0,
    }
)

In [72]:
import requests
try:
    req = requests.get(url="http://localhost:8000/health")
    if req.status_code == 200:
        print("✅ vLLM успешно подключён к LlamaIndex")
    else:
        print("Непоняточки какие-то!")
except Exception as e:
    print(f"Ошибка {e}")

✅ vLLM успешно подключён к LlamaIndex


In [73]:
local_data = Path.cwd() / "data"
# загружаем документы постранично
reader = SimpleDirectoryReader(input_dir=local_data, required_exts=[".pdf"])
docs = reader.load_data() # 86 docs = 86 lists of pdf

In [74]:
# для создания llm_questions
# splitter = TokenTextSplitter(chunk_size=14000, chunk_overlap=2000)
# nodes = splitter.get_nodes_from_documents(docs)

In [75]:
for i, doc in enumerate(docs):
    doc.metadata["page_label"] = i + 1

empty_indices = [i for i, doc in enumerate(docs) if len(doc.text) == 0]
print(f"Пустые страницы (реальные номера): {[docs[i].metadata['page_label'] for i in empty_indices]}")
for i in sorted(empty_indices, reverse=True):  
    docs.pop(i)

print(f"Документов после очистки: {len(docs)}")

Пустые страницы (реальные номера): [34, 86]
Документов после очистки: 84


In [90]:
docs[77].text

'№ Город Область\n219Кропоткин Краснодарский край\n220Крымск Краснодарский край\n221Лабинск Краснодарский край\n222Славянск-на-Кубани Краснодарский край\n223Тимашёвск Краснодарский край\n224Тихорецк Краснодарский край\n225Туапсе Краснодарский край\n226Зеленогорск Красноярский край\n227Лесосибирск Красноярский край\n228Минусинск Красноярский край\n229Назарово Красноярский край\n230Шадринск Курганская область\n231Кириши Ленинградская область\n232Апатиты Мурманская область\n233Североморск Мурманская область\n234Балахна Нижегородская область\n235Выкса Нижегородская область\n236Кстово Нижегородская область\n237Павлово Нижегородская область\n238Боровичи Новгородская область\n239Искитим Новосибирская область\n240Бугуруслан Оренбургская область\n241Бузулук Оренбургская область\n242Ливны Орловская область\n243Заречный Пензенская область\n244Краснокамск Пермский край\n245Кунгур Пермский край\n246Лысьва Пермский край\n247Арсеньев Приморский край\n248Горно-Алтайск Республика Алтай\n249Белебей Респ

In [77]:
# import asyncio

# async def generate_questions():
#     promt = """Твоя задача: прочитать текст и составить 2 вопроса (простой и сложный).
#     Твой ответ ДОЛЖЕН быть строго в формате JSON. Не пиши ничего, кроме JSON.

#     Формат ответа:
#     {"simple": "текст простого вопроса", "complex": "текст сложного вопроса"}

#     Текст для анализа:
#     """
    
#     # создаем список асинхронных задач
#     tasks = [Settings.llm.acomplete(prompt=promt + doc.text) for doc in docs]
    
#     responses = await asyncio.gather(*tasks)
#     return [r.text for r in responses]

# responses = await generate_questions()

In [78]:
import asyncio

async def generate_questions():
    promt = """Ты эксперт по договорным отношениям. Прочитай фрагмент оферты ПВЗ Wildberries.
        Составь 3 ВОПРОСА, которые мог бы задать владелец ПВЗ, ответ на которые 
        содержится ТОЛЬКО в этом фрагменте. Вопросы СТРОГО по оферте, НА ТЕМУ ОКАЗАНИЯ УСЛУГ.

        Пример плохого вопроса - Какая область относится к городу Волжский?
        Пример хороших вопросов: 
            - Какой срок ответа на претензию установлен в оферте?
            - Какой штраф налагается на Исполнителя за накопление мусора в ПВЗ после получения первого предупреждения?


        Уровни сложности:
        - simple: прямой простой вопрос,
        - medium: вопрос требует понимания контекста абзаца/пункта
        - complex: вопрос требует сопоставления нескольких пунктов внутри этого фрагмента

        Ответ строго в JSON:
        {"simple": "...", "medium": "...", "complex": "..."}

        Фрагмент:
        """
    
    # создаем список асинхронных задач
    tasks = [Settings.llm.acomplete(prompt=promt + doc.text + "\n /no_think") for doc in docs]
    
    responses = await asyncio.gather(*tasks)
    return [r.text for r in responses]

responses = await generate_questions()

2026-04-07 14:05:08,074 - INFO - HTTP Request: POST http://localhost:8000/v1/completions "HTTP/1.1 200 OK"
2026-04-07 14:05:08,464 - INFO - HTTP Request: POST http://localhost:8000/v1/completions "HTTP/1.1 200 OK"
2026-04-07 14:05:08,944 - INFO - HTTP Request: POST http://localhost:8000/v1/completions "HTTP/1.1 200 OK"
2026-04-07 14:05:09,585 - INFO - HTTP Request: POST http://localhost:8000/v1/completions "HTTP/1.1 200 OK"
2026-04-07 14:05:10,221 - INFO - HTTP Request: POST http://localhost:8000/v1/completions "HTTP/1.1 200 OK"
2026-04-07 14:05:10,222 - INFO - HTTP Request: POST http://localhost:8000/v1/completions "HTTP/1.1 200 OK"
2026-04-07 14:05:11,074 - INFO - HTTP Request: POST http://localhost:8000/v1/completions "HTTP/1.1 200 OK"
2026-04-07 14:05:11,432 - INFO - HTTP Request: POST http://localhost:8000/v1/completions "HTTP/1.1 200 OK"
2026-04-07 14:05:11,451 - INFO - HTTP Request: POST http://localhost:8000/v1/completions "HTTP/1.1 200 OK"
2026-04-07 14:05:12,856 - INFO - HTTP

In [79]:
len(responses)

84

In [80]:
responses[0]

'<think>\n\n</think>\n\n{\n  "simple": "Какое действие считается моментом Активации ПВЗ?",\n  "medium": "Что понимается под Блокировкой ПВЗ согласно оферте?",\n  "complex": "Какие действия могут привести к удержанию Товаров в качестве Подмен, и в течение какого срока Исполнитель может оспорить факт подмены?"\n}'

In [81]:
resp_end = []
for response in responses:
    resp_end.append(response.split("</think>")[1])

In [83]:
dict_to_json = {}
for resp, i in zip(resp_end, [doc.metadata["page_label"] for doc in docs]):
    resp.strip()
    dict_to_json[f"doc_{i}"] = resp.strip()
    # print(resp)
    # break

In [84]:
# dict_to_json

In [85]:
with open("docs_questions_qwen3_14b_awq.jsonl", "a", encoding="utf-8") as f:
    for k, v in dict_to_json.items():
        model_response_dict = json.loads(v)
        line_data = {k: model_response_dict}
        f.write(json.dumps(line_data, ensure_ascii=False) + "\n")